In [1]:
import os, cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from pathlib import Path
import shutil

In [2]:
ORIGINAL_DATASET_DIR = '/kaggle/input/datasets/almiraraisa/aptos-2019'
ORIGINAL_IMAGES = f'{ORIGINAL_DATASET_DIR}/train_images'
ORIGINAL_CSV = f'{ORIGINAL_DATASET_DIR}/train.csv'

OUTPUT_DIR = '/kaggle/working/aptos_2019_224pixels'
OUTPUT_IMAGES = f'{OUTPUT_DIR}/images'
os.makedirs(OUTPUT_IMAGES, exist_ok=True)

In [3]:
df = pd.read_csv(ORIGINAL_CSV)
df['id_code'] = df['id_code'].astype(str).apply(lambda x: x if x.endswith('.png') else x + '.png')

#1st split : 70% train + 30% temp
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['diagnosis'], random_state=42)

#2nd split: 15% val + 15% test
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['diagnosis'], random_state=42)

In [4]:
def print_proportions(dataframe, split_name):
    # Calculate the percentage of each class in the split
    proportions = dataframe['diagnosis'].value_counts(normalize=True).sort_index() * 100
    print(f"\n{split_name} Split ({len(dataframe)} images):")
    for cls, pct in proportions.items():
        print(f"  Class {cls}: {pct:.2f}%")

print_proportions(df, "Original Entire Dataset")
print_proportions(train_df, "Training")
print_proportions(val_df, "Validation")
print_proportions(test_df, "Testing")

train_df.to_csv(f'{OUTPUT_DIR}/train_split.csv', index=False)
val_df.to_csv(f'{OUTPUT_DIR}/val_split.csv', index=False)
test_df.to_csv(f'{OUTPUT_DIR}/test_split.csv', index=False)

print(f"Splits created and saved! Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}\n")


Original Entire Dataset Split (3662 images):
  Class 0: 49.29%
  Class 1: 10.10%
  Class 2: 27.28%
  Class 3: 5.27%
  Class 4: 8.06%

Training Split (2563 images):
  Class 0: 49.28%
  Class 1: 10.11%
  Class 2: 27.27%
  Class 3: 5.27%
  Class 4: 8.08%

Validation Split (549 images):
  Class 0: 49.36%
  Class 1: 10.02%
  Class 2: 27.32%
  Class 3: 5.28%
  Class 4: 8.01%

Testing Split (550 images):
  Class 0: 49.27%
  Class 1: 10.18%
  Class 2: 27.27%
  Class 3: 5.27%
  Class 4: 8.00%
Splits created and saved! Train: 2563, Val: 549, Test: 550



In [5]:
# resize image and remove black borders
def crop_and_resize(image_path, save_path, size=224):
    # Read image
    img = cv2.imread(str(image_path))
    if img is None: return
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
 
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        img = img[y:y+h, x:x+w]

    img_resized = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    
    # Save image
    cv2.imwrite(str(save_path), img_resized)

In [6]:
for img_name in tqdm(df['id_code'].values):
    src_path = os.path.join(ORIGINAL_IMAGES, img_name)
    dst_path = os.path.join(OUTPUT_IMAGES, img_name)
    crop_and_resize(src_path, dst_path, size=224)

print("\nPreprocessing complete.")

100%|██████████| 3662/3662 [09:39<00:00,  6.32it/s]


Preprocessing complete.


In [7]:
zip_path = '/kaggle/working/aptos_2019_224pixels'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"ZIP PATH: {zip_path}.zip")

ZIP PATH: /kaggle/working/aptos_2019_224pixels.zip
